# Exercise 02: Linear Regression – Rental Apartments in the Canton of Zurich

**Dataset:** Rental apartments in the canton of Zurich (ZHAW course data) – the running example of the slides
(rent vs. living area, prediction for a 95 m² flat, confidence vs. prediction interval, rent vs. rooms and area).
File: `../Data/apartments_data_enriched_cleaned.csv`.

**Business question:** How much more does a larger flat cost, how precisely can the rent of a flat be predicted, and
what is the role of the location?

**Slides:** Part 03 «Regression models» – linear regression (model, OLS, R², RMSE, confidence vs. prediction interval,
assumptions and diagnostics, multiple predictors, maximum likelihood).

**After this exercise you can ...**
- compute an OLS line, R² and RMSE by hand,
- fit and interpret simple and multiple linear regression models (*ceteris paribus*),
- check the model assumptions with residual plots,
- explain why "linear" means linear in the parameters (interaction terms),
- distinguish confidence and prediction intervals,
- explain the effect of correlated predictors (multicollinearity, VIF).

## Concept

**Model.** Multiple linear regression describes the target as a weighted sum of the features plus an error term:

$$y = \beta_0 + \beta_1 x_1 + \dots + \beta_p x_p + \varepsilon$$

Each $\beta_j$ is the change in the *average* $y$ per unit of $x_j$, **holding the other features constant**
(*ceteris paribus*) – an association, not automatically a causal effect. "Linear" means linear in the parameters:
$x^2$, $\log x$ or products $x_1 \cdot x_2$ (interactions) are allowed.

**OLS.** Ordinary least squares chooses the $\beta$'s that minimise the residual sum of squares
$\text{SSE}=\sum_i (y_i-\hat y_i)^2$. With one predictor:

$$\hat\beta_1 = \frac{\sum_i (x_i-\bar x)(y_i-\bar y)}{\sum_i (x_i-\bar x)^2}, \qquad \hat\beta_0 = \bar y - \hat\beta_1 \bar x$$

**Goodness of fit.**

$$R^2 = 1-\frac{\text{SSE}}{\text{SST}}, \quad \text{SST}=\sum_i (y_i-\bar y)^2, \qquad
\text{RMSE} = \sqrt{\tfrac{1}{n}\,\text{SSE}}$$

$R^2$ = share of the variation of $y$ explained by the model; RMSE = typical prediction error in the units of $y$.

**Confidence vs. prediction interval.** The 95 % *confidence interval* covers the **average** $y$ of all observations with
the given $x$; the 95 % *prediction interval* covers the $y$ of **one new** observation and is always wider.

**Assumptions** (check with residual plots): linearity, independence, constant variance (homoscedasticity), normality of
the errors (for exact small-sample inference), no perfect multicollinearity (check with the VIF).

**Maximum likelihood.** Choose the parameters under which the observed data are most plausible. For independent normal
errors with constant variance, ML gives exactly the OLS solution.

## Libraries and settings

In [ ]:
# Libraries
import os
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.model_selection import train_test_split
from sklearn.metrics import root_mean_squared_error, r2_score

# Ignore warnings
import warnings
warnings.filterwarnings('ignore')

# Show current working directory
print(os.getcwd())

## Exercise 1: OLS by hand

Five rental flats:

| living area $x$ [m²] | 40 | 60 | 80 | 100 | 120 |
|---|---|---|---|---|---|
| rent $y$ [CHF/month] | 1200 | 1800 | 2100 | 2700 | 3200 |

a) Compute $\bar x$, $\bar y$, the slope $\hat\beta_1$ and the intercept $\hat\beta_0$ **by hand**.

b) Compute the fitted values, the residuals, SSE, SST, $R^2$ and the RMSE.

c) Interpret $\hat\beta_1$ and $\hat\beta_0$. What is the predicted rent for a flat with 95 m² (as on the slides)?

d) Verify your results with `numpy`.

In [ ]:
x = np.array([40, 60, 80, 100, 120])
y = np.array([1200, 1800, 2100, 2700, 3200])

beta_1 = np.sum((x - x.mean()) * (y - y.mean())) / np.sum((x - x.mean())**2)
beta_0 = y.mean() - beta_1 * x.mean()
y_hat  = beta_0 + beta_1 * x
res    = y - y_hat
sse    = np.sum(res**2)
sst    = np.sum((y - y.mean())**2)

print('x_mean =', x.mean(), ' y_mean =', y.mean())
print('beta_1 =', round(beta_1, 3), ' beta_0 =', round(beta_0, 3))
print('fitted values:', y_hat.round(1))
print('residuals:    ', res.round(1))
print('SSE =', round(sse, 1), ' SST =', round(sst, 1))
print('R2 =', round(1 - sse/sst, 4), ' RMSE =', round(np.sqrt(sse/len(y)), 1))
print('Prediction for 95 m2:', round(beta_0 + beta_1 * 95, 1))

# Check with numpy's polyfit
print('np.polyfit:', np.polyfit(x, y, deg=1).round(3))

**Solution:**

a) $\bar x = 80$, $\bar y = 2200$. $\sum(x_i-\bar x)(y_i-\bar y) = 40'000+8'000+0+10'000+40'000 = 98'000$,
$\sum(x_i-\bar x)^2 = 1600+400+0+400+1600 = 4000$ → $\hat\beta_1 = 24.5$, $\hat\beta_0 = 2200-24.5\cdot 80 = 240$.

b) Fitted values 1220, 1710, 2200, 2690, 3180; residuals −20, 90, −100, 10, 20;
SSE = 400+8'100+10'000+100+400 = 19'000; SST = 1'000'000+160'000+10'000+250'000+1'000'000 = 2'420'000;
$R^2 = 1-19'000/2'420'000 \approx 0.992$; RMSE $=\sqrt{19'000/5} \approx 61.6$ CHF.

c) Per additional m², the rent is on average 24.50 CHF per month higher. The intercept (240 CHF for a flat with 0 m²)
lies outside the data and has no substantive meaning. Prediction for 95 m²: $240 + 24.5\cdot 95 = 2'567.50$ CHF.

## The apartment data

The slides use rents of flats as the running example (rent vs. living area, prediction for a 95 m² flat, confidence vs.
prediction interval, rent vs. rooms and area). In this exercise we repeat these steps with **real** rental listings.

**Dataset:** 774 rental apartments in the canton of Zurich, scraped from an online real estate portal and enriched with
municipality data (ZHAW course data).

| variable | meaning |
|---|---|
| `rooms` | number of rooms |
| `area` | living area (m²) |
| `price` | monthly rent (CHF) |
| `bfs_name` | municipality |
| `pop_dens`, `mean_taxable_income`, `dist_supermarket`, … | municipality and location data |

## Exercise 2: Import and explore the apartment data

a) Import the apartment data (`sep=';'`) and keep the columns `bfs_name`, `rooms`, `area` and `price`. Remove duplicates
and missing values, and keep only flats with a rent between 1'000 and 5'000 CHF (as in the other course notebooks).

b) Create a scatter plot of `price` against `area`. Compare it with the (simulated) data on the slides.

c) Split the data into a training set (80 %) and a test set (20 %) with `random_state=42`.

In [ ]:
# a) Import and clean the data
apt = pd.read_csv('../Data/apartments_data_enriched_cleaned.csv', sep=';')
apt = apt[['bfs_name', 'rooms', 'area', 'price']].drop_duplicates().dropna()
apt = apt[(apt['price'] >= 1000) & (apt['price'] <= 5000)].reset_index(drop=True)
print(apt.shape)
print(apt.describe().round(1))

# b) Scatter plot price vs. area
plt.figure(figsize=(7, 4))
plt.scatter(apt['area'], apt['price'], s=10, alpha=0.5)
plt.xlabel('Living area [m²]')
plt.ylabel('Rent [CHF/month]')
plt.title('Rental apartments in the canton of Zurich')
plt.grid(alpha=0.3)
plt.show()

# c) Train/test split of the data frame
apt_train, apt_test = train_test_split(apt, test_size=0.20, random_state=42)
print('Training set:', apt_train.shape, ' Test set:', apt_test.shape)

**Solution:**

a) After cleaning, 714 flats remain (living area 15–270 m², median ≈ 86 m²; rent median ≈ 2'350 CHF).

b) Rent clearly increases with living area, but the scatter is much larger than on the slides: flats of the same size
cost very different amounts depending on location (city of Zurich vs. countryside), age and fittings. The slide data were
simulated to lie almost perfectly on a line; real data are noisier.

## Exercise 3: Simple linear regression – rent vs. living area

a) Fit the model `price ~ area` on the training set with `smf.ols()` and print the summary.

b) Interpret $\hat\beta_0$, $\hat\beta_1$ and $R^2$. Compare $\hat\beta_1$ with the slope on the slides
(30.84 CHF per m²).

c) Predict the rent of a 95 m² flat and compute the RMSE on the test set.

In [ ]:
# a) Simple linear regression
apt_model_1 = smf.ols('price ~ area', data=apt_train).fit()
print(apt_model_1.summary())

# b) Prediction for 95 m2 and test RMSE
flat_95 = pd.DataFrame({'area': [95]})
print('Predicted rent for 95 m2:', round(apt_model_1.predict(flat_95).iloc[0], 2), 'CHF')
print('Test RMSE:', round(root_mean_squared_error(apt_test['price'], apt_model_1.predict(apt_test))), 'CHF')

**Solution:**

$\hat\beta_1 \approx 13.9$: per additional m² the rent is on average about 13.90 CHF higher – less than half of the
slope on the slides (30.84). $\hat\beta_0 \approx 1'250$ CHF is the (theoretical) rent for 0 m²; it has no substantive
meaning, but it shows that small flats are relatively expensive per m². $R^2 \approx 0.37$: living area explains only
about a third of the variation of the rents – much less than for the simulated slide data.

A 95 m² flat is predicted to cost about **2'570 CHF** (slides: 3'097 CHF). On the test set the typical error is about
**700 CHF** – living area alone is not enough to predict rents accurately.

## Exercise 4: Confidence vs. prediction interval (95 m²)

a) Compute the 95 % confidence and prediction intervals for a flat with 95 m²
(`apt_model_1.get_prediction(...).summary_frame(alpha=0.05)`).

b) Plot the regression line with both intervals for living areas from 20 to 200 m² into the scatter plot of the training
data (as on the slides).

c) Interpret both intervals. Why is the prediction interval so much wider than the confidence interval? Compare with the
slides (CI ≈ 3'000–3'140 CHF, PI ≈ 2'620–3'520 CHF).

In [ ]:
# a) Intervals for 95 m2
pred_95 = apt_model_1.get_prediction(pd.DataFrame({'area': [95]})).summary_frame(alpha=0.05)
print(pred_95.round(0).to_string())

# b) Plot with confidence and prediction intervals
area_grid = pd.DataFrame({'area': np.linspace(20, 200, 200)})
bands = apt_model_1.get_prediction(area_grid).summary_frame(alpha=0.05)

plt.figure(figsize=(8, 5))
plt.scatter(apt_train['area'], apt_train['price'], s=8, alpha=0.4, color='grey', label='training data')
plt.fill_between(area_grid['area'], bands['obs_ci_lower'], bands['obs_ci_upper'],
                 color='tab:orange', alpha=0.25, label='95% prediction interval')
plt.fill_between(area_grid['area'], bands['mean_ci_lower'], bands['mean_ci_upper'],
                 color='tab:blue', alpha=0.4, label='95% confidence interval')
plt.plot(area_grid['area'], bands['mean'], color='tab:blue', linewidth=2, label='OLS line')
plt.xlabel('Living area [m²]')
plt.ylabel('Rent [CHF/month]')
plt.legend(loc='upper left')
plt.grid(alpha=0.3)
plt.show()

**Solution:**

For 95 m²: **CI ≈ 2'510–2'620 CHF**, **PI ≈ 1'320–3'810 CHF**.

The confidence interval says where the *average* rent of all 95 m² flats lies. It is narrow, because the line is
estimated from almost 600 flats. The prediction interval says where the rent of *one* new 95 m² flat will lie with 95 %
probability. It also contains the large scatter of individual flats around the line (location, fittings, …) and is
therefore much wider – here about ±1'250 CHF, compared to ±450 CHF on the slides, because the real data are much
noisier. The confidence interval becomes wider towards the edges of the data (few very small or very large flats).

## Exercise 5: Multiple regression – rooms and area (ceteris paribus)

a) Fit `price ~ rooms` and `price ~ rooms + area` on the training set. Compare the coefficient of `rooms` in both models.

b) Interpret the coefficients of the model with both predictors (*ceteris paribus*). Why does the coefficient of `rooms`
change so much? (Hint: compute the correlation between `rooms` and `area`.)

c) Compute the variance inflation factor (VIF) of `rooms` and `area` (add a constant with `sm.add_constant()` first).
What would happen if you added the variable `area_in_dm2 = 100 * area`?

d) Compare with the slides (rooms: +36.48 CHF, area: +9.80 CHF per m²).

In [ ]:
# a) Models with rooms
apt_model_r  = smf.ols('price ~ rooms', data=apt_train).fit()
apt_model_ra = smf.ols('price ~ rooms + area', data=apt_train).fit()
print('price ~ rooms:        ', apt_model_r.params.round(1).to_dict())
print(apt_model_ra.summary().tables[1])
print('R2 (rooms + area):', round(apt_model_ra.rsquared, 3), '  R2 (area only):', round(apt_model_1.rsquared, 3))

# b) Correlation between rooms and area
print('\nCorrelation rooms - area:', round(apt_train[['rooms', 'area']].corr().iloc[0, 1], 2))

# c) VIF
X_vif = sm.add_constant(apt_train[['rooms', 'area']])
vif = pd.DataFrame({'feature': X_vif.columns,
                    'VIF': [variance_inflation_factor(X_vif.values, i) for i in range(X_vif.shape[1])]})
print(vif[vif['feature'] != 'const'].round(2).to_string(index=False))

**Solution:**

a) Alone, each additional room is associated with about **+330 CHF** rent. In the model with both predictors the
coefficient of `rooms` becomes **negative** (≈ −54 CHF) and is not significant (p ≈ 0.2), while the coefficient of `area`
stays similar (≈ 15.5 CHF per m²). $R^2$ hardly increases (0.368 → 0.369).

b) *Ceteris paribus:* "one room more **at the same living area**" means that the existing space is divided into more
(smaller) rooms – this does not make a flat more expensive. Flats with more rooms are expensive mainly because they are
larger. `rooms` and `area` are strongly correlated (r ≈ 0.86), so once `area` is in the model, `rooms` contains hardly any
additional information.

c) VIF ≈ 3.8 for both predictors: noticeable but not critical correlation (rule of thumb: problematic above 5–10). The
standard error of the `rooms` coefficient is inflated, which is one reason why it is not significant. `area_in_dm2` would
be a perfect linear function of `area` (**perfect multicollinearity**): the coefficients would not be identifiable, the VIF
would be infinite.

d) On the slides both coefficients are positive, because the simulated data were generated that way. With real data the
interpretation of multiple regression coefficients requires care: each coefficient answers a *ceteris paribus* question,
which may differ from the simple "the more rooms, the more expensive".

## Exercise 6: Residual diagnostics

a) Plot the residuals of `apt_model_1` (`price ~ area`) against the fitted values.

b) Create a Q-Q plot of the residuals (`sm.qqplot(..., line='45', fit=True)`).

c) Which assumptions of the slides («Linear regression: assumptions and diagnostics») are violated?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# a) Residuals vs. fitted values
axes[0].scatter(apt_model_1.fittedvalues, apt_model_1.resid, s=10, alpha=0.5)
axes[0].axhline(0, color='black', linewidth=1)
axes[0].set_xlabel('Fitted values [CHF]')
axes[0].set_ylabel('Residuals [CHF]')
axes[0].set_title('Residuals vs. fitted (price ~ area)')

# b) Q-Q plot
sm.qqplot(apt_model_1.resid, line='45', fit=True, ax=axes[1])
axes[1].set_title('Q-Q plot of residuals')
plt.show()

# Spread of the residuals for small and large fitted values
print(apt_model_1.resid.groupby(pd.cut(apt_model_1.fittedvalues, [0, 2000, 2500, 3000, 6000])).std().round(0))

**Solution:**

- **Equal variance (homoscedasticity) is violated:** the spread of the residuals increases with the fitted values
  (larger flats → larger absolute deviations; standard deviation of the residuals ≈ 550 CHF for small vs. ≈ 680 CHF for
  large flats). Prediction intervals are therefore too wide for small flats and too narrow for large flats.
- **Normality is violated in the upper tail:** the Q-Q plot shows more large positive residuals than a normal
  distribution would produce (a few flats are much more expensive than their size suggests – e.g. in the city centre).
- **Linearity** is roughly fulfilled; there is no clear curved pattern. However, the large residuals hint at an important
  missing predictor: the **location**.

## Exercise 7: Location and interaction – "linear" means linear in the parameters

Flats in the **city of Zurich** are more expensive than in the rest of the canton. Is the difference a fixed amount, or
does each m² cost more in the city?

$$\text{price} = \beta_0 + \beta_1\,\text{area} + \beta_2\,\text{city} + \beta_3\,(\text{area}\cdot\text{city}) + \varepsilon$$

a) Create the dummy variable `city` (1 if `bfs_name == 'Zürich'`, else 0) in the training and test set. Which share of
the flats is located in the city?

b) Fit `price ~ area + city` and `price ~ area * city` (the `*` adds both main effects and the interaction) on the
training set.

c) Interpret the coefficients of the interaction model: what does one additional m² cost in the city and outside? How
did $R^2$ change compared to `price ~ area`?

d) Plot the two fitted lines (city / rest of the canton) into a scatter plot coloured by `city`.

In [ ]:
# a) Dummy variable city
for d in [apt_train, apt_test]:
    d['city'] = (d['bfs_name'] == 'Zürich').astype(int)
print('Share of flats in the city of Zurich:', round(apt_train['city'].mean(), 3))

# b) Models with city
apt_model_c  = smf.ols('price ~ area + city', data=apt_train).fit()
apt_model_ci = smf.ols('price ~ area * city', data=apt_train).fit()
print(apt_model_ci.summary().tables[1])

# c) Price per additional m2 in the city and outside
p = apt_model_ci.params
print('Additional m2 outside the city:', round(p['area'], 2), 'CHF')
print('Additional m2 in the city:     ', round(p['area'] + p['area:city'], 2), 'CHF')
print('R2:  area', round(apt_model_1.rsquared, 3), '| area + city', round(apt_model_c.rsquared, 3),
      '| area * city', round(apt_model_ci.rsquared, 3))

# d) Scatter plot with two lines
area_grid = pd.DataFrame({'area': np.linspace(20, 200, 100)})
plt.figure(figsize=(8, 5))
for c, label, color in [(0, 'rest of the canton', 'tab:blue'), (1, 'city of Zurich', 'tab:red')]:
    sub = apt_train[apt_train['city'] == c]
    plt.scatter(sub['area'], sub['price'], s=10, alpha=0.4, color=color, label=label)
    plt.plot(area_grid['area'], apt_model_ci.predict(area_grid.assign(city=c)), color=color, linewidth=2)
plt.xlabel('Living area [m²]')
plt.ylabel('Rent [CHF/month]')
plt.title('price ~ area * city')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

**Solution:**

a) About 29 % of the flats in the training set are located in the city of Zurich.

c) Interaction model: outside the city one additional m² costs on average about **15.90 CHF** per month, in the city
about **22.10 CHF** (15.89 + 6.25). In addition, city flats have a higher base level (+370 CHF). The interaction is
significant (p < 0.001): the location changes not only the level but also the **slope**. $R^2$ increases from 0.37
(`area`) to 0.57 (`area + city`) and 0.59 (`area * city`) – the location explains a large part of the variation that
living area alone cannot explain. The model is still *linear in the parameters*; the product `area · city` is just
another column of the design matrix.

d) The red line (city) lies above the blue line (rest of the canton) and is steeper.

## Exercise 8: Evaluation on the test set

So far $R^2$ was computed on the training data. Compute **RMSE** and **$R^2$ on the test set** for the models
`price ~ area`, `price ~ rooms + area`, `price ~ area + city` and `price ~ area * city` and summarise the results in a
table. Which model do you recommend?

In [ ]:
results = []
for name, m in [('price ~ area', apt_model_1),
                ('price ~ rooms + area', apt_model_ra),
                ('price ~ area + city', apt_model_c),
                ('price ~ area * city', apt_model_ci)]:
    y_pred = m.predict(apt_test)
    results.append({'model': name,
                    'R2 (train)': round(m.rsquared, 3),
                    'R2 (test)': round(r2_score(apt_test['price'], y_pred), 3),
                    'RMSE (test)': round(root_mean_squared_error(apt_test['price'], y_pred))})
print(pd.DataFrame(results).to_string(index=False))

**Solution:**

| model | test $R^2$ | test RMSE |
|---|---|---|
| `price ~ area` | ≈ 0.31 | ≈ 705 CHF |
| `price ~ rooms + area` | ≈ 0.30 | ≈ 710 CHF |
| `price ~ area + city` | ≈ 0.60 | ≈ 540 CHF |
| `price ~ area * city` | ≈ 0.60 | ≈ 537 CHF |

`rooms` does not improve the predictions (it is almost redundant given `area`, see Exercise 5). The location variable
`city` reduces the test RMSE by about a quarter. The interaction brings only a small additional gain on the test set;
both city models are good choices – `price ~ area * city` is recommended if the different price per m² is of interest.
Training and test $R^2$ are similar → no sign of overfitting. Even the best model is off by about 540 CHF on average –
more information (fittings, floor, exact location) would be needed for precise rent predictions.

## Exercise 9 (optional): Maximum likelihood

The share of flats located in the city of Zurich is to be estimated. Suppose in a random sample of $n=10$ flats,
$k=3$ are located in the city. For a probability $p$ the likelihood of this sample (in a given order) is
$L(p) = p^{k}(1-p)^{n-k}$.

a) Compute $L(p)$ for $p = 0.1, 0.3, 0.5$ by hand.

b) Plot $L(p)$ for $p \in [0, 1]$ and find the maximum. Compare with the share of city flats in the full data set.

In [ ]:
n, k = 10, 3

def likelihood(p):
    return p**k * (1 - p)**(n - k)

for p in [0.1, 0.3, 0.5]:
    print(f'L({p}) = {likelihood(p):.5f}')

p_grid = np.linspace(0, 1, 501)
p_hat = p_grid[np.argmax(likelihood(p_grid))]

plt.figure(figsize=(6, 3.5))
plt.plot(p_grid, likelihood(p_grid))
plt.axvline(p_hat, color='darkred', linestyle='--', label=f'maximum at p = {p_hat:.2f}')
plt.xlabel('p')
plt.ylabel('L(p)')
plt.title('Likelihood of 3 city flats in 10 flats')
plt.legend()
plt.show()

print('Share of city flats in the full data set:', round((apt['bfs_name'] == 'Zürich').mean(), 3))

**Solution:**

a) $L(0.1) = 0.1^3 \cdot 0.9^7 \approx 0.00048$, $L(0.3) = 0.3^3 \cdot 0.7^7 \approx 0.00222$,
$L(0.5) = 0.5^{10} \approx 0.00098$.

b) The maximum lies at $\hat p = k/n = 0.3$ – the ML estimate is simply the observed share. In the full data set about
30 % of the flats are located in the city of Zurich. The same principle (choose the parameters that make the data most
plausible) is used to fit logistic regression (Exercise 04); for linear regression with normal errors it gives exactly
the OLS solution.

### Jupyter notebook --footer info-- (please always provide this at the end of each notebook)

In [ ]:
import os
import platform
import socket
from platform import python_version
from datetime import datetime

print('-----------------------------------')
print(os.name.upper())
print(platform.system(), '|', platform.release())
print('Datetime:', datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print('Python Version:', python_version())
print('-----------------------------------')